## Introduction 
Higher education has long been understood as a stepping stone to prepare young adults to enter the workforce. More specifically to teach and enhance skills necessary for the desired role of each student. However, a [Hult International Business School](https://www.hult.edu/blog/wi_skills_survey/) conducted a survey that reports students feel as though they are lacking skills when they enter the workforce. Less than a quarter of college graduates said that they lacked skills necessary for their current jobs. Over 77% graduates also say they learned more in 6 months of their jobs than in their entire undergraduate experience. Lastly, 96% of HR leaders believe that schools need to do a better job in workforce training. So, if students and employers feel dissatisfied with how undergraduate education is preparing people for future jobs, is there a skill gap between educational curriculums and skills required for jobs? 
	Our study aims to hone into how Cornell courses are bridging, or failing to bridge, the gap between the skills taught at Cornell and the skills necessary for data jobs. Specifically, data scientists, data analysts, and data engineers in the year 2023. We are utilizing course roster data from the Cornell course roster API. Specifically filtering for skills mentioned in the course description. We are also using Luke Barousse’s public 2023 job-posting dataset for job required skills. The data sets will be described in more detail in the data description section of this report. 

### Research Questions 
1. How do the technical skills that employers request differ between US data analyst, data scientist and data engineer job postings in 2023?
2. How often are those skills mentioned in Cornell's computing and data related course descriptions from the same year?



## Data Description and Cleaning

### Job-posting data

We used Luke Barousse’s data_jobs.csv dataset which contains online job postings that are collected through Google from several job sites. Each row represents one job posting. We used the posting date, country, standardized job title and list of skills. The dataset uses a model to standardize the job titles and extracts skill names from the postings.

We kept the postings from 2023 that were labeled United States and had a standardized job title of Data Analyst, Data Scientist or Data Engineer. We then removed exact duplicate rows and excluded postings without a usable skill list. If a job posting listed Python (or any other term) twice then we counted it as one posting that listed Python. After cleaning, we had 141,910 postings. For each role- we calculated the percentage mentioning a skill by dividing the number of postings that listed that skill by the total number of postings for that role with a usable skill list.

Barousse collected job listings through Google which means that the dataset may miss jobs advertised offline or listings that didn’t appear in his search results. The dataset may miss some skills listed in a posting or put a posting in the wrong job category.

[Hugging Face dataset and documentation](https://huggingface.co/datasets/lukebarousse/data_jobs) · [Original CSV file](https://huggingface.co/datasets/lukebarousse/data_jobs/tree/main)

### Cornell course data

We used the Cornell Class Roster API to collect course descriptions from Spring and Fall 2023. We included four subjects because they cover computing, information science, statistics and quantitative modeling: Computer Science (CS), Information Science (INFO), Statistics and Data Science (STSCI), and Operations Research and Information Engineering (ORIE). We decided to select courses by their subject codes before checking which skills appeared in their descriptions.

We kept courses with descriptions that were not blank and counted each course ID once across both semesters. If a course had 2 different descriptions then we searched both and counted the course once if either description mentioned the skill. After cleaning, we had 398 distinct courses.

## Revised Hypotheses and Analysis Plan

In Phase 2- our first analysis had 11 yearly observations and our second had 10 year-to-year changes. After we got feedback we decided to change our question to compare specific skills in 2023 job postings and Cornell course descriptions. We had already made an exploratory table of the 20 most common skills before writing this revised plan.

### Hypothesis 1

We expect the Data Engineer postings to list docker more often than Data Analyst postings. Docker helps package software so that it can run consistently in different environments which is relevant to building and running data pipelines.

**Planned analysis:** For each posting we will record if it lists docker. We will compare the percentages across the 3 job roles and use logistic regression to test if the odds of listing docker are higher for Data Engineer postings compared to Data Analyst postings.

### Hypothesis 2

We expect skills that appear more often in job postings to also appear more often in Cornell course descriptions. Tools that are useful for many kinds of data work may come up in both job postings and courses.

**Planned analysis:** For each of the 20 skills we will calculate the percent of job postings across all 3 roles that list the skill and the percent of Cornell courses whose descriptions mention it. We will use Spearman correlation to measure if skills that rank higher in job postings also tend to rank higher in Cornell course descriptions.

We will first calculate the correlation by pairing each skill’s job posting percent with its own Cornell percent. Then we will keep the job posting percents in place and randomly shuffle the Cornell percentages 10,000 times. we will count how many times shuffling gives us a correlation = to or greater than the correlation we got when we matched each skill to itself.

Before choosing this test, we had already looked at the job posting and Cornell percents for all 20 skills. We treat the test as exploratory because it measures a pattern in data that we had already seen.

In [ ]:
# load the 2 cleaned datasets and turn the saved job skills into lists.
import ast
import pandas as pd

jobs = pd.read_csv("job_postings_2023_clean.csv")
courses = pd.read_csv("cornell_courses_2023_clean.csv")

# the csv stores each skill list as text. Convert it back to a list.
jobs["job_skills"] = jobs["job_skills"].map(ast.literal_eval)

print("Job postings:", len(jobs))
print("Cornell courses:", len(courses))

Job postings: 141910
Cornell courses: 398


### How we matched skills across the datasets

We selected the 20 skills that appeared in the most job postings across the three roles. We used the skill lists provided in the job dataset and searched the Cornell course descriptions for those same skills. A course counted once for a skill even if its description mentioned it several times.

In [ ]:
# Find the 20 most common skills and see how often each job role lists them.
postings_per_role = jobs["job_title_short"].value_counts()
job_skills = jobs.explode("job_skills", ignore_index=True)
# Select the 20 most common skills across all three roles.
top_skills = job_skills["job_skills"].value_counts().head(20).index

skill_counts = pd.crosstab(
    job_skills["job_skills"], job_skills["job_title_short"]
).reindex(top_skills)

skill_percentages = (
    skill_counts.div(postings_per_role, axis="columns") * 100
).round(1)

skill_percentages

job_title_short,Data Analyst,Data Engineer,Data Scientist
job_skills,,,
sql,60.0,71.9,58.1
python,32.0,68.2,82.0
r,20.3,9.5,50.4
tableau,33.7,15.1,26.9
excel,47.9,6.0,12.8
aws,5.8,45.0,19.9
spark,2.4,33.7,19.1
azure,5.0,33.8,13.3
power bi,20.2,10.4,11.0


In [ ]:
# Count how many cornell course descriptions mention each of the 20 skills.
import re

# Use more specific patterns where a skill name can mean something else.
special_patterns = {
    "r": r"\bR\b",
    "excel": r"\bExcel\b",
    "word": r"(?i)\b(?:Microsoft|MS)\s+Word\b",
    "spark": r"(?i)\b(?:Apache\s+Spark|PySpark)\b|\bSpark\b",
    "azure": r"(?i)\bMicrosoft\s+Azure\b|\bAzure\b",
    "oracle": r"\bOracle\b",
    "snowflake": r"\bSnowflake\b",
"sql": r"(?i)\bSQL\b(?!\s+Server\b|[-–—]\s*like\b)",}

descriptions = courses["description"].fillna("")
cornell_counts = {}

for skill in top_skills:
    pattern = special_patterns.get(skill, rf"(?i)\b{re.escape(skill)}\b")
    cornell_counts[skill] = descriptions.str.contains(pattern).sum()

comparison = skill_percentages.copy()
comparison["Cornell course count"] = pd.Series(cornell_counts)
comparison["Cornell courses (%)"] = (
    comparison["Cornell course count"] / len(courses) * 100
).round(1)

comparison

job_title_short,Data Analyst,Data Engineer,Data Scientist,Cornell course count,Cornell courses (%)
job_skills,,,,,
sql,60.0,71.9,58.1,2,0.5
python,32.0,68.2,82.0,16,4.0
r,20.3,9.5,50.4,26,6.5
tableau,33.7,15.1,26.9,0,0.0
excel,47.9,6.0,12.8,3,0.8
aws,5.8,45.0,19.9,0,0.0
spark,2.4,33.7,19.1,1,0.3
azure,5.0,33.8,13.3,0,0.0
power bi,20.2,10.4,11.0,0,0.0


### Exploratory comparison of job postings and course descriptions

SQL appeared in at least 58% of postings with usable skill lists in each of the three roles. Other skills varied more by role. Excel appeared in 47.9% of data analyst postings, compared with 6.0% of data engineer postings and 12.8% of data scientist postings. Python appeared in 82.0% of data scientist postings, compared with 68.2% of data engineer postings and 32.0% of data analyst postings.

Among the 398 Cornell courses we examined, 26 descriptions mentioned R (6.5%), 16 mentioned Python (4.0%), and 2 mentioned SQL (0.5%). The two sources describe different things. Job postings list skills employers ask for, while short course descriptions may leave out tools students use in class. These results tell us which skills were explicitly mentioned in each source.

In [ ]:
# Read the words around each R match to check that it means the language
for course_id, description in zip(courses["crseId"], descriptions):
    for match in re.finditer(special_patterns["r"], description):
        start = max(0, match.start() - 60)
        end = match.end() + 60
        snippet = description[start:end].replace("\n", " ")
        print(course_id, snippet)

352214 me securities, and resampling techniques. Covers the use of R for statistical calculations, simulation, and optimization.
352376 odeling and parallel computing. Programming will be done in R.
358529 ming (VBA), and programming in a scripting language (either R or Python). Operations research methods include inventory m
358808 les of statistical programming (such as data frames, Python/R packages, reproducible workflows, and version control), uni
359479  bayesian statistics. Students are instructed in the use of R software; prior knowledge of R is helpful but not required.
359479 are instructed in the use of R software; prior knowledge of R is helpful but not required. This course is intended for M.
360952  to linear modeling. Applied analysis is carried out in the R statistical computing environment.
364083 rs. Homework assignments will involve computation using the R statistical package.
364513 methods. Some assignments may involve computation using the R programming language.


In [ ]:
# Show the words around Excel and Spark so we can check what they mean.
for skill in ["excel", "spark"]:
    pattern = special_patterns[skill]

    for course_id, description in zip(courses["crseId"], descriptions):
        for match in re.finditer(pattern, description):
            start = max(0, match.start() - 60)
            end = match.end() + 60
            snippet = description[start:end].replace("\n", " ")
            print(skill, course_id, snippet)

excel 358529 ery language (SQL), geographical information systems (GIS), Excel and Visual Basic programming (VBA), and programming in a sc
excel 359480 sion problems in a hands-on learning environment. Microsoft Excel is heavily used. A wide variety of application areas are co
excel 371823 sion problems in a hands-on learning environment. Microsoft Excel is heavily used. A wide variety of application areas are co
spark 367928 NumPy/Scipy, scikit-learn), parallel computing (Map-Reduce, Spark, Hadoop), basic finite-precision arithmetic, an overview of


### Checking the skill matches

We read the words around each R match in the Cornell course descriptions. All 37 matches referred to the R programming language and came from 26 distinct courses. Some descriptions mentioned R more than once. We also checked that the Excel matches referred to the spreadsheet software and that the Spark match referred to the tool used for parallel computing.

For job postings, we used the skill lists that came with the dataset. We spot-checked the Docker indicator by comparing it with the saved skill lists. The CSV does not include the original posting text, so this check only tells us whether our code counted the saved labels correctly. We also counted postings that listed Python, R or Excel directly from the saved skill lists. For each skill those counts matched our comparison table in all 3 job roles.

In [ ]:
# mark each posting with a 1 if it lists Docker and a 0 if it doesnt.
jobs["lists_docker"] = [
    int("docker" in skills) for skills in jobs["job_skills"]
]

jobs[["job_title_short", "job_skills", "lists_docker"]].head()

,job_title_short,job_skills,lists_docker
0,Data Analyst,"[jira, python, sql]",0
1,Data Scientist,"[docker, python, r, splunk, tableau]",1
2,Data Scientist,"[alteryx, cognos, python, r, spss, sql, tableau]",0
3,Data Scientist,"[excel, python, r, sql, tableau]",0
4,Data Engineer,"[azure, bash, databricks, excel, git, hadoop, ...",0


In [ ]:
# Find the % of postings in each job role that list Docker.
docker_by_role = jobs.groupby("job_title_short")["lists_docker"].agg(
    total_postings="size",
    postings_with_docker="sum"
)

docker_by_role["percent_with_docker"] = (
    100 * docker_by_role["postings_with_docker"]
    / docker_by_role["total_postings"]
).round(1)

docker_by_role

,total_postings,postings_with_docker,percent_with_docker
job_title_short,,,
Data Analyst,57411,305,0.5
Data Engineer,32814,2488,7.6
Data Scientist,51685,2791,5.4


In [ ]:
# Compare the odds of listing Docker across job roles using logistic regression.
import statsmodels.formula.api as smf

# Predict whether a posting lists Docker using its job role.
# Use Data Analyst as the reference group.
docker_model = smf.logit(
    'lists_docker ~ C(job_title_short, Treatment(reference="Data Analyst"))',
    data=jobs
).fit(disp=False)

docker_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           lists_docker   No. Observations:               141910
Model:                          Logit   Df Residuals:                   141907
Method:                           MLE   Df Model:                            2
Date:                Fri, 06 Nov 2026   Pseudo R-squ.:                 0.08359
Time:                        13:52:59   Log-Likelihood:                -21571.
converged:                       True   LL-Null:                       -23539.
Covariance Type:            nonrobust   LLR p-value:                     0.000
=============================================================================================================================================
                                                                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                    -5.2324      0.057    -91.136      0.000      -5.345      -5.120
C(job_title_short, Treatment(reference="Data Analyst"))[T.Data Engineer]      2.7318      0.061     44.723      0.000       2.612       2.852
C(job_title_short, Treatment(reference="Data Analyst"))[T.Data Scientist]     2.3691      0.061     39.080      0.000       2.250       2.488
=============================================================================================================================================
"""

### Hypothesis 1 Results

We predicted that Data Engineer postings would list docker more often than Data Analyst postings. Out of the postings with usable skill lists 7.6% of Data Engineer postings listed Docker, compared with 0.5% of Data Analyst postings and 5.4% of Data Scientist postings.

We used job role to predict the probability that a posting listed Docker. Using Data Analyst as the reference group the model estimated that the odds of listing Docker were about 15.4 times as high for Data Engineer postings as for Data Analyst postings. Even in the role with the highest rate- fewer than 8 in 100 postings listed Docker.

The p value for the comparison between Data Engineer and Data Analyst postings was below 0.001. We rejected the null hypothesis that the two roles had equal odds of listing Docker. This supports our prediction for the 2023 postings with usable skill lists in our dataset.

In [ ]:
# Put the job posting and Cornell course percentages side by side for each skill.
skill_comparison = pd.DataFrame({
    "job_postings_pct": 100 * skill_counts.sum(axis=1)/len(jobs),
    "cornell_courses_pct": 100 * comparison["Cornell course count"]/len(courses)})

skill_comparison.head().round(1)

,job_postings_pct,cornell_courses_pct
job_skills,,
sql,62.1,0.5
python,58.6,4.0
r,28.8,6.5
tableau,26.9,0.0
excel,25.4,0.8


In [ ]:
# Use spearman correlation to see how closely the two skill rankings agree.
job_percentages = skill_comparison["job_postings_pct"]
cornell_percentages = skill_comparison["cornell_courses_pct"]

observed_correlation = job_percentages.corr(
    cornell_percentages, method="spearman"
)

print(f"Spearman correlation: {observed_correlation:.3f}")

Spearman correlation: 0.457


In [ ]:
# shuffle the Cornell percentages to see how often we get a correlation this high.
import numpy as np
from scipy.stats import spearmanr

rng = np.random.default_rng(2950)
num_shuffles = 10_000
at_least_as_high = 0

for _ in range(num_shuffles):
    # Keep the job percentages in place and rearrange the Cornell percentages.
    shuffled_cornell = rng.permutation(cornell_percentages)
    shuffled_correlation = spearmanr(job_percentages, shuffled_cornell)[0]

    if shuffled_correlation >= observed_correlation - 1e-12:
        at_least_as_high += 1

# Include the observed result in the count.
p_value = (at_least_as_high + 1) / (num_shuffles + 1)

print(f"Shuffles at least as high as the observed correlation: {at_least_as_high:,}")
print(f"One-sided p-value: {p_value:.4f}")

Shuffles at least as high as the observed correlation: 227
One-sided p-value: 0.0228


### Hypothesis 2 Results

We predicted that skills listed more often in job postings would also be mentioned more often in Cornell course descriptions. Across the 20 skills, the Spearman correlation was 0.457. This means skills that ranked higher in job postings tended to rank higher in Cornell course descriptions, although the rankings differed. For example, SQL ranked higher than R in job postings, while R ranked higher than SQL in Cornell course descriptions.

In 227 out of 10,000 random pairings, the correlation was at least as high as the one we observed. The one-sided p-value was 0.0228. Since this is below 0.05, we rejected the null hypothesis that the skill rankings were unrelated. This result supports our prediction for the 20 skills we studied. We treat it as exploratory because we had already looked at the skill comparison table before choosing this test.